# A02 — P+7 (Oulipian language modelling)

**CART 498 — Generative AI**

A variation on the Oulipo's **N+7** technique. Instead of replacing each noun with the noun seven entries later in a dictionary, **P+7** replaces the **last word of each line** of Wallace Stevens' *The Snow Man* with the token GPT-2 ranks **7th most probable** at that position.

**P+x** generalises this: the replacement is GPT-2's *x*-th most probable token.

### How it works
For each line:
1. Split off the last word (keeping its punctuation, e.g. `snow;` keeps `;`).
2. Give GPT-2 the original poem up to that point plus the line without its last word.
3. Rank every possible next token by probability.
4. Keep only tokens that are complete English words (checked against the NLTK word list), so we don't replace a word with `,`, a line break or a fragment like `gro`.
5. Take the token at rank *x* and put it in place of the original word.

Run the cells from top to bottom (**Runtime → Run all**). The `.txt` files are written to an `A02/` folder (open the folder icon in the left sidebar to download them).

## 1. Setup

In [ ]:
# transformers and torch are pre-installed in Google Colab.
# If you run this somewhere else, uncomment the next line:
# !pip install transformers torch

import os
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
model.eval()

OUT_DIR = "A02"
os.makedirs(OUT_DIR, exist_ok=True)
print("Loaded", MODEL_ID)

## 2. The poem

In [ ]:
TITLE = "The Snow Man"
AUTHOR = "Wallace Stevens"

poem = """One must have a mind of winter
To regard the frost and the boughs
Of the pine-trees crusted with snow;
And have been cold a long time
To behold the junipers shagged with ice,
The spruces rough in the distant glitter
Of the January sun; and not to think
Of any misery in the sound of the wind,
In the sound of a few leaves,
Which is the sound of the land
Full of the same wind
That is blowing in the same bare place
For the listener, who listens in the snow,
And, nothing himself, beholds
Nothing that is not there and the nothing that is."""

lines = poem.split("\n")
print(len(lines), "lines")

## 3. The P+x functions

`WORDS_ONLY = True` keeps only tokens that are complete English words (a leading space + a word found in the NLTK word list).
Set it to `False` to use the raw x-th token, even if it is punctuation, a newline or a word fragment.

In [ ]:
WORDS_ONLY = True

def ranked_next_tokens(context, top_n=3000):
    """GPT-2's top_n next tokens for `context`, most probable first, as (text, probability)."""
    input_ids = tokenizer(context, return_tensors="pt").input_ids
    with torch.no_grad():
        logits = model(input_ids).logits[0, -1, :]
    probs = torch.softmax(logits, dim=-1)
    top_probs, top_ids = torch.topk(probs, top_n)
    return [(tokenizer.decode([i]), p.item()) for i, p in zip(top_ids.tolist(), top_probs)]


import nltk
nltk.download("words", quiet=True)
from nltk.corpus import words as nltk_words
ENGLISH = {w.lower() for w in nltk_words.words()} | {w.lower() for w in re.findall(r"[A-Za-z]+", poem)}

SHORT_WORDS = {"a", "i", "am", "an", "as", "at", "be", "by", "do", "go", "he", "if", "in", "is", "it",
               "me", "my", "no", "of", "oh", "on", "or", "so", "to", "up", "us", "we"}

def is_word_token(tok):
    """True for tokens like ' winter': a leading space + a complete English word.
    This rejects punctuation, line breaks and word fragments such as ' gro' or ' breat'."""
    w = tok[1:]
    if not (tok.startswith(" ") and w.isalpha()):
        return False
    if len(w) <= 2:                      # the word list contains fragments like 'th', 'sh'
        return w.lower() in SHORT_WORDS
    return w.lower() in ENGLISH


def split_last_word(line):
    """'Of the pine-trees crusted with snow;' -> ('Of the pine-trees crusted with', 'snow', ';')"""
    m = re.match(r"^(.*?)\s*([\w'-]+)(\W*)$", line)
    return m.group(1), m.group(2), m.group(3)


def p_plus_x(lines, x=7, words_only=WORDS_ONLY, verbose=True):
    """Replace the last word of every line with GPT-2's x-th most probable next token."""
    new_lines, log = [], []
    for i, line in enumerate(lines):
        prefix, word, punct = split_last_word(line)

        # Context = original poem so far + current line without its last word
        context = "\n".join(lines[:i] + [prefix])

        candidates = ranked_next_tokens(context)
        if words_only:
            candidates = [(t, p) for t, p in candidates if is_word_token(t)]

        new_tok, prob = candidates[x - 1]          # rank x -> index x-1
        new_word = new_tok.strip()
        new_lines.append(f"{prefix} {new_word}{punct}".strip())
        log.append((word, new_word, prob))

        if verbose:
            print(f"{word:>10}  ->  {new_word:<14} rank {x:<3} p = {prob*100:6.3f}%")
    return new_lines, log


def save_version(new_lines, x, filename=None):
    filename = filename or f"P+{x}.txt"
    path = os.path.join(OUT_DIR, filename)
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"{TITLE} (P+{x})\n")
        f.write(f"after {AUTHOR}, transformed with GPT-2\n\n")
        f.write("\n".join(new_lines) + "\n")
    print("Saved", path)

## 4. Sanity check: P+1

With x = 1 GPT-2 picks its single most likely word, so this shows what the model "expects" at the end of each line.

In [ ]:
p1, _ = p_plus_x(lines, x=1)
print()
print("\n".join(p1))

## 5. P+7

In [ ]:
p7, p7_log = p_plus_x(lines, x=7)
print()
print("\n".join(p7))
save_version(p7, 7)

## 6. Exploring x

Generate many versions to compare. Read them and pick the funniest, wittiest or most absurd one.

In [ ]:
X_VALUES = [2, 3, 5, 10, 15, 20, 25, 30, 42, 50, 75, 100, 250, 500]

versions = {}
for x in X_VALUES:
    versions[x], _ = p_plus_x(lines, x=x, verbose=False)
    print(f"\n===== P+{x} =====")
    print("\n".join(versions[x]))

# Also save every version in one file for easy comparison (outside the A02 folder)
with open("all_x_versions.txt", "w", encoding="utf-8") as f:
    for x, v in versions.items():
        f.write(f"===== P+{x} =====\n" + "\n".join(v) + "\n\n")

### Side-by-side view of the replaced words

In [ ]:
originals = [split_last_word(l)[1] for l in lines]
header = f"{'original':<10}" + "".join(f"{'P+'+str(x):<13}" for x in [1, 7] + X_VALUES[:6])
print(header)
print("-" * len(header))
table_versions = {1: p1, 7: p7, **versions}
for row, orig in enumerate(originals):
    cells = [split_last_word(table_versions[x][row])[1] for x in [1, 7] + X_VALUES[:6]]
    print(f"{orig:<10}" + "".join(f"{c:<13}" for c in cells))

## 7. P+x — my chosen version

✏️ **Change `BEST_X`** to the value you found funniest, then run this cell.

In [ ]:
BEST_X = 250   # <-- my chosen x

px, px_log = p_plus_x(lines, x=BEST_X)
print()
print("\n".join(px))
save_version(px, BEST_X)

## 8. (Optional) Sketch: P+7 for every noun

The reflection asks how to replace **all nouns** with their 7th most probable alternative. A possible implementation:

1. Find nouns with a part-of-speech tagger (spaCy).
2. For each noun, give GPT-2 the text *before* it.
3. Walk down GPT-2's ranking and keep only candidates that spaCy also tags as a noun in that position.
4. Take the 7th such noun.

Limitation: GPT-2 only sees the left context. A masked model such as BERT could use the words on both sides.

In [ ]:
import spacy
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")


def noun_p_plus_x(text, x=7, max_checks=400):
    doc = nlp(text)
    replacements = []                       # (start_char, end_char, new_word)
    for tok in doc:
        if tok.pos_ != "NOUN":
            continue
        context = text[:tok.idx].rstrip(" ")
        found = 0
        for cand, p in ranked_next_tokens(context)[:max_checks]:
            if not is_word_token(cand) or cand.strip().lower() == tok.text.lower():
                continue
            trial = nlp(context + cand)
            if trial[-1].pos_ == "NOUN":
                found += 1
                if found == x:
                    replacements.append((tok.idx, tok.idx + len(tok.text), cand.strip()))
                    print(f"{tok.text:>10}  ->  {cand.strip()}")
                    break
    # Replace from the end so character positions stay valid
    out = text
    for start, end, new in reversed(replacements):
        out = out[:start] + new + out[end:]
    return out


all_nouns_p7 = noun_p_plus_x(poem, x=7)
print()
print(all_nouns_p7)

## 9. Download everything

Run this to download the `.txt` files (also available from the folder icon on the left).
Download the notebook itself with **File → Download → Download .ipynb**.

In [ ]:
try:
    from google.colab import files
    for name in sorted(os.listdir(OUT_DIR)):
        files.download(os.path.join(OUT_DIR, name))
except ImportError:
    print("Not in Colab — files are in", os.path.abspath(OUT_DIR))